### 🎯 Surrogate Model Extraction (Black-Box Evasion)

In this notebook, we will train a **Surrogate Model** to extract the decision boundary of a black-box IDS. 
We do not have access to the dataset, parameters, or type of the defending model. We can only interact with it by querying an API with simulated packet characteristics and receiving a response (`1` for normal, `-1` for anomaly).

In [1]:
import numpy as np
import pandas as pd
import random
import joblib
from sklearn.tree import DecisionTreeClassifier

# Setting random seeds for reproducibility
random.seed(42)
np.random.seed(42)

In [2]:
def preprocess_data(df):
    df = df.copy()
    df = pd.get_dummies(df, columns=['protocol'], drop_first=True)
    if 'protocol_UDP' not in df.columns:
        df['protocol_UDP'] = 0
    columns = ['src_port', 'dst_port', 'packet_size', 'duration_ms', 'protocol_UDP']
    return df[columns]


def query_defender_api(packet_size, duration_ms, src_port=443, protocol="TCP"):
    """
    Simulates a query to the remote black-box IDS.
    Returns:
       1 if the defender classifies the packet as NORMAL.
      -1 if the defender classifies the packet as ANOMALY.
    """
    defender_model = joblib.load("anomaly_model.joblib")

    query_df = pd.DataFrame([{
        "src_port": src_port,
        "dst_port": 80,
        "packet_size": packet_size,
        "duration_ms": duration_ms,
        "protocol": protocol
    }])

    query_array = preprocess_data(query_df)
    prediction = defender_model.predict(query_array)[0]
    return prediction

In [3]:
# Generate random probe inputs to query the black box and build our surrogate dataset.
# We will generate probe samples covering different ranges of packet sizes and durations.

num_probes = 400
probe_data = []

for _ in range(num_probes):
    packet_size = random.randint(50, 4000)
    duration_ms = random.randint(10, 2000)
    probe_data.append({
        "packet_size": packet_size,
        "duration_ms": duration_ms
    })

df_probes = pd.DataFrame(probe_data)

In [4]:
labels = []

for _, row in df_probes.iterrows():
    label = query_defender_api(row['packet_size'], row['duration_ms'])
    labels.append(label)

df_probes['label'] = labels
display(df_probes.head())

,packet_size,duration_ms,label
0,2669,238,1
1,152,1528,1
2,1176,511,1
3,964,295,1
4,3066,219,1


In [5]:
X_surrogate = df_probes[['packet_size', 'duration_ms']]
y_surrogate = df_probes['label']

surrogate_model = DecisionTreeClassifier(random_state=42)
surrogate_model.fit(X_surrogate, y_surrogate)

accuracy = surrogate_model.score(X_surrogate, y_surrogate)
print(f"Surrogate model approximation accuracy: {accuracy * 100:.2f}%")

Surrogate model approximation accuracy: 100.00%


In [6]:
# Save the surrogate model, so it can be used by the adversary server
joblib.dump(surrogate_model, "surrogate_model.joblib")
print("Surrogate model saved as surrogate_model.joblib")

Surrogate model saved as surrogate_model.joblib
